# Homework

In [22]:
import os
import pickle
import click
import pandas as pd
import numpy as np

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction import DictVectorizer

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/sklearn/metrics/__init__.py)

In [24]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope

In [25]:
from pathlib import Path

ROOT = Path.cwd()
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "nyc-taxi-experiment"
RAW_DATA_PATH = ROOT / "data"
OUTPUT_PATH = ROOT / "output"
PREPROCESS_SCRIPT = ROOT / "preprocess_data.py"
TRAIN_SCRIPT = ROOT / "train.py"

print(f"Working directory: {ROOT}")
print(f"Tracking URI: {TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Preprocess script path: {PREPROCESS_SCRIPT}")
print(f"Train script path: {TRAIN_SCRIPT}")

if not PREPROCESS_SCRIPT.exists() or not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(
        "Missing preprocess_data.py or train.py. Run the Script Sources cells first."
    )

Working directory: /workspaces/mlops-zoomcamp/02-experiment-tracking/Homework
Tracking URI: http://127.0.0.1:5000
Experiment: nyc-taxi-experiment
Preprocess script path: /workspaces/mlops-zoomcamp/02-experiment-tracking/Homework/preprocess_data.py
Train script path: /workspaces/mlops-zoomcamp/02-experiment-tracking/Homework/train.py


Q1. Install MLflow

In [18]:
import mlflow

mlflow.__version__

'1.25.1'

Answer: 1.25.1

Q2. Download and preprocess the data

In [26]:
from importlib import reload
import preprocess_data

reload(preprocess_data)
preprocess_data.run_data_prep.callback(
    raw_data_path=str(RAW_DATA_PATH),
    dest_path=str(OUTPUT_PATH),
    dataset="green",
)

print("Preprocess completed successfully.")

Preprocess completed successfully.


Answer: 4

Q3. Train a model with autolog

In [27]:
import train

os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT_NAME

reload(train)
train.run_train.callback(data_path=str(OUTPUT_PATH))

print("Training completed and run logged to MLflow.")

2026/07/06 11:05:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


RMSE: 5.431162180141208
Training completed and run logged to MLflow.


Answer: 2

Q4 Launch the tracking server locally

Answer: default-artifact-root

In [ ]:
from importlib import reload
import hpo

reload(hpo)
hpo.run_optimization.callback(data_path=str(OUTPUT_PATH), num_trials=15)

print("Hyperparameter optimization completed and logged to MLflow.")

## Scripts

In [ ]:
#Preprocess data
def dump_pickle(obj, filename: str):
    with open(filename, "wb") as f_out:
        return pickle.dump(obj, f_out)


def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df: pd.DataFrame, dv: DictVectorizer, fit_dv: bool = False):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    if fit_dv:
        X = dv.fit_transform(dicts)
    else:
        X = dv.transform(dicts)
    return X, dv


@click.command()
@click.option(
    "--raw_data_path",
    help="Location where the raw NYC taxi trip data was saved"
)
@click.option(
    "--dest_path",
    help="Location where the resulting files will be saved"
)
def run_data_prep(raw_data_path: str, dest_path: str, dataset: str = "green"):
    # Load parquet files
    df_train = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-01.parquet")
    )
    df_val = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-02.parquet")
    )
    df_test = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-03.parquet")
    )

    # Extract the target
    target = 'duration'
    y_train = df_train[target].values
    y_val = df_val[target].values
    y_test = df_test[target].values

    # Fit the DictVectorizer and preprocess data
    dv = DictVectorizer()
    X_train, dv = preprocess(df_train, dv, fit_dv=True)
    X_val, _ = preprocess(df_val, dv, fit_dv=False)
    X_test, _ = preprocess(df_test, dv, fit_dv=False)

    # Create dest_path folder unless it already exists
    os.makedirs(dest_path, exist_ok=True)

    # Save DictVectorizer and datasets
    dump_pickle(dv, os.path.join(dest_path, "dv.pkl"))
    dump_pickle((X_train, y_train), os.path.join(dest_path, "train.pkl"))
    dump_pickle((X_val, y_val), os.path.join(dest_path, "val.pkl"))
    dump_pickle((X_test, y_test), os.path.join(dest_path, "test.pkl"))


if __name__ == '__main__':
    run_data_prep()

Overwriting preprocess_data.py


In [ ]:
#Train data
def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
def run_train(data_path: str):
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
    experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "nyc-taxi-experiment")

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)
    mlflow.autolog()

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    with mlflow.start_run():
        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)
        print(f"RMSE: {rmse}")

if __name__ == '__main__':
    run_train()

Overwriting train.py


In [ ]:
%%writefile hpo.py
import os
import pickle
import click
import mlflow
import numpy as np

from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-hyperopt")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
@click.option(
    "--num_trials",
    default=15,
    help="The number of parameter evaluations for the optimizer to explore"
)
def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):
        with mlflow.start_run(nested=True):
            mlflow.log_params(params)

            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = mean_squared_error(y_val, y_pred, squared=False)

            mlflow.log_metric("rmse", rmse)
            return {"loss": rmse, "status": STATUS_OK}

    search_space = {
        "max_depth": scope.int(hp.quniform("max_depth", 1, 20, 1)),
        "n_estimators": scope.int(hp.quniform("n_estimators", 10, 50, 1)),
        "min_samples_split": scope.int(hp.quniform("min_samples_split", 2, 10, 1)),
        "min_samples_leaf": scope.int(hp.quniform("min_samples_leaf", 1, 4, 1)),
        "random_state": 42,
    }

    rstate = np.random.default_rng(42)
    trials = Trials()
    with mlflow.start_run(run_name="hyperopt-parent") as parent_run:
        best_result = fmin(
            fn=objective,
            space=search_space,
            algo=tpe.suggest,
            max_evals=num_trials,
            trials=trials,
            rstate=rstate,
        )

        best_rmse = min(trials.losses())
        mlflow.log_metric("best_rmse", best_rmse)
        mlflow.log_metric("num_trials", num_trials)
        mlflow.log_params({f"best_{k}": v for k, v in best_result.items()})
        print("Parent run:", parent_run.info.run_id)
        print("Best RMSE:", best_rmse)
        print("Best result:", best_result)


if __name__ == '__main__':
    run_optimization()

2026/07/06 11:10:55 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.


100%|██████████| 15/15 [01:42<00:00,  6.83s/trial, best loss: 5.335419588556921]
Parent run: 04cf1c91ccba4e68bfc7e93e75d31cee
Best result: {'max_depth': 19.0, 'min_samples_leaf': 2.0, 'min_samples_split': 2.0, 'n_estimators': 11.0}
